In [61]:
import os
from langchain_core.tools import tool
from langchain_community.document_loaders import WebBaseLoader

from dotenv import load_dotenv
from spotipy import Spotify
from spotipy.exceptions import SpotifyException
from spotipy.oauth2 import SpotifyOAuth
from spotipy.cache_handler import CacheFileHandler

In [62]:
load_dotenv()

SPOTIFY_CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
SPOTIFY_CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")
scopes = "streaming playlist-read-private playlist-modify-private user-top-read user-read-recently-played user-library-modify user-library-read user-read-currently-playing app-remote-control user-modify-playback-state user-read-playback-state"
redirect_uri = "https://127.0.0.1:8000/callback"
cache_handler = CacheFileHandler(cache_path='cache')

spotify_oauth = SpotifyOAuth(client_id=SPOTIFY_CLIENT_ID, client_secret=SPOTIFY_CLIENT_SECRET, redirect_uri=redirect_uri, scope=scopes, show_dialog=True, cache_handler=cache_handler)
sp = Spotify(auth_manager=spotify_oauth)


In [ ]:
import json


def get_user_playlists():
    try:
        playlists = sp.current_user_playlists()
    except SpotifyException as e:
        raise RuntimeError(f"Spotify error while fetching playlists: {e}") from e
    except Exception as e:
        raise RuntimeError(f"Unexpected error while fetching playlists: {e}") from e

    if not playlists or not playlists.get('items'):
        raise LookupError("No playlists found for this user.")

    return playlists['items']


json.dumps(get_user_playlists(), indent=4)



TypeError: get_user_playlists() missing 1 required positional argument: 'self'

In [88]:
def play_next_song():
    devices = sp.devices().get('devices', [])
    if not devices:
        raise RuntimeError("No Spotify device found. Open Spotify first.")

    try:
        sp.next_track()
    except SpotifyException as e:
        raise RuntimeError(f"Spotify error while skipping to next song: {e}") from e
    except Exception as e:
        raise RuntimeError(f"Unexpected error while skipping to next song: {e}") from e

    return "Skipped to the next song."

play_next_song()

'Skipped to the next song.'

In [80]:
def play_previous_song():
    devices = sp.devices().get('devices', [])
    if not devices:
        raise RuntimeError("No Spotify device found. Open Spotify first.")

    try:
        sp.previous_track()
    except SpotifyException as e:
        raise RuntimeError(f"Spotify error while going to previous song: {e}") from e
    except Exception as e:
        raise RuntimeError(f"Unexpected error while going to previous song: {e}") from e

    return "Went back to the previous song."

play_previous_song()


'Went back to the previous song.'

In [84]:
def play_this_song(song_name):
    if not song_name or not song_name.strip():
        raise ValueError("Please provide a valid song name.")

    try:
        results = sp.search(q=song_name, type='track')
        items = results.get('tracks', {}).get('items', [])
        if not items:
            raise LookupError(f"No track found for '{song_name}'.")

        uri = items[0]['uri']
        track_name = items[0].get('name', song_name)

        devices = sp.devices().get('devices', [])
        if not devices:
            raise RuntimeError("No Spotify device found. Open Spotify first.")

        device_id = devices[0]['id']
        sp.start_playback(device_id=device_id, uris=[uri])
        return f"Playing {track_name}."
    except SpotifyException as e:
        raise RuntimeError(f"Spotify error while playing song: {e}") from e

play_this_song("")

ValueError: Please provide a valid song name.

In [85]:
def play_my_playlist(playlist_name):
    if not playlist_name or not playlist_name.strip():
        raise ValueError("Please provide a valid playlist name.")

    try:
        playlists = sp.current_user_playlists()
        items = playlists.get('items', []) if playlists else []
        if not items:
            raise LookupError("You don't have any playlists.")

        for playlist in items:
            if playlist['name'].lower() == playlist_name.lower():
                uri = playlist['uri']

                devices = sp.devices().get('devices', [])
                if not devices:
                    raise RuntimeError("No Spotify device found. Open Spotify first.")

                device_id = devices[0]['id']

                sp.transfer_playback(device_id=device_id, force_play=True)
                sp.start_playback(device_id=device_id, context_uri=uri)
                return f"Playing {playlist['name']}"

        raise LookupError(f"Playlist '{playlist_name}' not found in your library.")
    except SpotifyException as e:
        raise RuntimeError(f"Spotify error while playing playlist: {e}") from e

play_my_playlist("Yaaa")

LookupError: Playlist 'Yaaa' not found in your library.

In [86]:
def play_playlist(playlist_name):
    if not playlist_name or not playlist_name.strip():
        raise ValueError("Please provide a valid playlist name.")

    try:
        results = sp.search(q=playlist_name, type='playlist')
        items = results.get('playlists', {}).get('items', []) if results else []
        items = [p for p in items if p]
        if not items:
            raise LookupError(f"No playlist found for '{playlist_name}'.")

        uri = items[0]['uri']
        name = items[0].get('name', playlist_name)

        devices = sp.devices().get('devices', [])
        if not devices:
            raise RuntimeError("No Spotify device found. Open Spotify first.")

        device_id = devices[0]['id']
        sp.start_playback(device_id=device_id, context_uri=uri)
        return f"Playing playlist '{name}'."
    except SpotifyException as e:
        raise RuntimeError(f"Spotify error while playing playlist: {e}") from e

play_playlist("ya")

'Playing playlist \'Ya Ali - From "Zubeen Garg"\'.'

In [161]:
def pause_song():
        try:
            sp.pause_playback()
        except SpotifyException as e:
            return f"Spotify error while pausing song: {e}"
        except Exception as e:
            return f"Unexpected error while pausing song: {e}"

        return "Paused the song."

pause_song()

'Paused the song.'

In [166]:
def resume_song():
        try:
            sp.start_playback()
        except SpotifyException as e:
            return f"Spotify error while resuming song: {e}"
        except Exception as e:
            return f"Unexpected error while resuming song: {e}"

        return "Resumed the song."

resume_song()


'Resumed the song.'

In [87]:
def add_in_queue(song_name):
    if not song_name or not song_name.strip():
        raise ValueError("Please provide a valid song name.")

    try:
        results = sp.search(q=song_name, type='track')
        items = results.get('tracks', {}).get('items', []) if results else []
        if not items:
            raise LookupError(f"No track found for '{song_name}'.")

        uri = items[0]['uri']
        track_name = items[0].get('name', song_name)

        devices = sp.devices().get('devices', [])
        if not devices:
            raise RuntimeError("No Spotify device found. Open Spotify first.")

        device_id = devices[0]['id']
        sp.add_to_queue(uri, device_id=device_id)
        return f"Added '{track_name}' to the queue."
    except SpotifyException as e:
        raise RuntimeError(f"Spotify error while adding to queue: {e}") from e

add_in_queue("Shape of You")

"Added 'Shape of You' to the queue."

In [ ]:
def remove_from_queue(song_name):
    if not song_name or not song_name.strip():
        raise ValueError("Please provide a valid song name.")

    try:
        queue_data = sp.queue()
    except SpotifyException as e:
        raise RuntimeError(f"Spotify error while reading queue: {e}") from e

    queue_items = queue_data.get('queue', []) if queue_data else []
    if not queue_items:
        raise LookupError("Queue is empty.")

    normalized = song_name.strip().lower()
    matching_item = next(
        (track for track in queue_items if track.get('name', '').strip().lower() == normalized),
        None,
    )

    if not matching_item:
        raise LookupError(f"'{song_name}' is not currently in the queue.")

    raise NotImplementedError(
        "Spotify Web API does not support removing a specific song from queue."
    )

In [92]:
devices = sp.devices().get('devices', [])
json.dumps(devices, indent=2)

'[\n  {\n    "id": "7875a8b3733ccab9911ae18bf5c0384f5cc9584c",\n    "is_active": true,\n    "is_private_session": false,\n    "is_restricted": false,\n    "name": "MacBook Pro",\n    "supports_volume": true,\n    "type": "Computer",\n    "volume_percent": 100\n  }\n]'

In [ ]:
import subprocess
import webbrowser
subprocess.run(['open', '-a', 'Google Chrome'])



True

In [158]:
def open_browser(browser_name: str = "Safari", url: str = "https://www.google.com"):

    subprocess.run(['open', '-a', browser_name, url])

open_browser("Google Chrome")

In [101]:
from pathlib import Path

def create_folder(folder_name: str) -> str:
    if not folder_name or not folder_name.strip():
        return "Please provide a valid folder name."
    else:
        base_path = Path.home()
        full_path = base_path / folder_name
        Path(full_path).mkdir(parents=True, exist_ok=True)
        return f"Created folder: {full_path}"

create_folder("")


'Please provide a valid folder name.'

In [102]:
def create_file(file_name: str) -> str:
    if not file_name or not file_name.strip():
        return "Please provide a valid file name."
    else:
        base_path = Path.home()
        full_path = base_path / file_name
        Path(full_path).touch(exist_ok=True)
        return f"Created file: {full_path}"

create_file("Documents/test/test.txt")

'Created file: /Users/niloysaha/Documents/test/test.txt'

In [113]:
import subprocess
import os

def open_folder(folder_path: str) -> str:
    if not folder_path or not folder_path.strip():
        return "Please provide a valid folder path."
    
    # Expand ~ and convert to absolute path
    folder_path = os.path.abspath(os.path.expanduser(folder_path))

    if not os.path.exists(folder_path):
        return f"Path does not exist: {folder_path}"

    subprocess.run(['open', folder_path])
    return f"Opened folder: {folder_path}"

open_folder("~/Documents/test")


'Opened folder: /Users/niloysaha/Documents/test'

In [114]:
import os
print(os.path.isdir("/Users/niloysaha/Documents/test"))
print(os.listdir("/Users/niloysaha/Documents"))

True
['Songs', 'Unreal Projects', '.DS_Store', 'test', '.localized', 'constructionWebsite', 'CSC249Assignment', '.tmp.drivedownload', 'Documents - Niloy’s MacBook Air', 'Library', "Documents - Niloy's MacBook Air", 'Zoom', 'lecture2_intro_nlp.key', 'priceScraper']


In [115]:
def open_file(file_path: str) -> str:
    if not file_path or not file_path.strip():
        return "Please provide a valid file path."
    
    # Expand ~ and convert to absolute path
    file_path = os.path.abspath(os.path.expanduser(file_path))

    if not os.path.exists(file_path):
        return f"Path does not exist: {file_path}"

    subprocess.run(['open', file_path])
    return f"Opened file: {file_path}"

open_file("~/Documents/test/test.txt")

import os
print(os.path.isfile("/Users/niloysaha/Documents/test/test.txt"))


True


In [ ]:
def open_folder(folder_path: str, app: str) -> str:
    if not folder_path or not folder_path.strip():
        return "Please provide a valid folder path."
    
    # Expand ~ and convert to absolute path
    folder_path = os.path.abspath(os.path.expanduser(folder_path))

    if not os.path.exists(folder_path):
        return f"Path does not exist: {folder_path}"

    subprocess.run(['open', '-a', app, folder_path])
    return f"Opened folder: {folder_path}"

open_folder("~/Documents/test", "Finder")


'Opened folder: /Users/niloysaha/Documents/test'

In [ ]:
def rename_folder_file(folder_path: str, new_name: str) -> str:
    if not folder_path or not folder_path.strip():
        return "Please provide a valid folder path."
    if not new_name or not new_name.strip():
        return "Please provide a valid new name."
    else:
        folder_path = os.path.abspath(os.path.expanduser(folder_path))
        new_path = os.path.join(os.path.dirname(folder_path), new_name)
        os.rename(folder_path, new_path)
        return f"Renamed folder: {folder_path} to {new_name}"

rename_folder("~/Documents/test2/test2.txt ", "test2.txt")


'Renamed folder: /Users/niloysaha/Documents/test2/test2.txt  to test2.txt'

In [ ]:
import shutil

def copy_file(file_path: str, new_path: str) -> str:
        if not file_path or not file_path.strip():
            return "Please provide a valid folder path."
        if not new_path or not new_path.strip():
            return "Please provide a valid new path."
        else:
            file_path = os.path.abspath(os.path.expanduser(file_path))
            new_path = os.path.abspath(os.path.expanduser(new_path))
            if not os.path.exists(file_path) and not os.path.exists(new_path):
                return f"Path does not exist: {file_path}"
            shutil.copy(file_path, new_path)
            return f"Copied file: {file_path} to {new_path}"

copy_file("~/Documents/test2/test2.txt", "~/Documents/priceScraper")

'Copied file: /Users/niloysaha/Documents/test2/test2.txt to /Users/niloysaha/Documents/priceScraper'

In [151]:
def copy_folder(folder_path: str, new_path: str) -> str:
    if not folder_path or not folder_path.strip():
        return "Please provide a valid folder path."
    if not new_path or not new_path.strip():
        return "Please provide a valid new path."
    else:
        folder_path = os.path.abspath(os.path.expanduser(folder_path))
        new_path = os.path.abspath(os.path.expanduser(new_path))
        if not os.path.exists(folder_path) or not os.path.exists(new_path):
            return f"Path does not exist: {folder_path}"
        final_dest = os.path.join(new_path, os.path.basename(folder_path))
        shutil.copytree(folder_path, final_dest, dirs_exist_ok=True)
        return f"Copied folder: {folder_path} to {new_path}"

copy_folder("~/Documents/test2", "~/Documents/a")

'Copied folder: /Users/niloysaha/Documents/test2 to /Users/niloysaha/Documents/a'

In [4]:
from newsapi import NewsApiClient
from dotenv import load_dotenv
from datetime import datetime
import os

load_dotenv()

news_api_key = os.getenv("NEWS_API_KEY")

news_api = NewsApiClient(api_key=news_api_key)

def get_news(query: str):
    return news_api.get_everything(q=query, sort_by='relevancy', language='en', to=datetime.now().strftime('%Y-%m-%d'))

get_news("bitcoin")

{'status': 'ok',
 'totalResults': 5133,
 'articles': [{'source': {'id': None, 'name': 'Gizmodo.com'},
   'author': 'Kyle Torpey',
   'title': 'Popular Musician Loses Life Savings Through Malicious Crypto Wallet in Apple’s App Store',
   'description': 'The theft apparently combined a counterfeit app with a critical mistake by the musician.',
   'url': 'https://gizmodo.com/popular-musician-loses-life-savings-through-malicious-crypto-wallet-in-apples-app-store-2000745902',
   'urlToImage': 'https://gizmodo.com/app/uploads/2026/04/g-love-crypto-theft-1200x675.jpg',
   'publishedAt': '2026-04-13T21:25:17Z',
   'content': 'Musician G. Love lost his life savings after downloading a fake Ledger Live app from Apple’s Mac App Store, according to a post made to his X account. Noted blockchain investigator ZachXBT traced the… [+4424 chars]'},
  {'source': {'id': None, 'name': 'BBC News'},
   'author': None,
   'title': 'Lib Dems call for inquiry into Farage Bitcoin deal',
   'description': 'The R